In [1]:
import logging
import time
import os
import pickle

#from DataPrep import data_prep

import numpy as np
import matplotlib.pyplot as plt

#import tensorflow_datasets as tfds
import tensorflow as tf

# Import tf_text to load the ops used by the tokenizer saved model
#import tensorflow_text  # pylint: disable=unused-import
import pandas as pd
import numpy as np
import re
import seaborn as sns
import matplotlib as plt


from sklearn.model_selection import train_test_split


from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Model,  Sequential
from tensorflow.keras.layers import LSTM, GRU, Bidirectional, Dropout, Input, TimeDistributed, Dense, Activation, RepeatVector, Embedding, Concatenate
import tensorflow.keras.layers as layers
from tensorflow.keras.layers import Attention
from tensorflow.keras.optimizers import Adam, Adagrad
from keras.losses import sparse_categorical_crossentropy
logging.getLogger('tensorflow').setLevel(logging.ERROR)  # suppress warnings
import random

In [2]:
with open("Pichia_TrTs_2Target.pkl", "rb") as fp:
    Data_AllOrg = pickle.load(fp)
    
AA_tr = Data_AllOrg['AA_tr']
Cds_tr = Data_AllOrg['Cds_tr']

In [3]:
Settings = pd.read_csv('../BO_forHyperParameter/Arch1/Round2.csv').iloc[:, 1:]
Settings

,Enc hidden size,Enc Embedding size,Dec Embedding size,Dense Layer size,Dense Layer size aa,Drop rate,Drop rate aa
0,60.0,235.0,130.0,30.0,199.0,0.0,0.0
1,60.0,151.0,148.0,30.0,152.0,0.9,0.0
2,193.0,83.0,215.0,260.0,152.0,0.9,0.0


#### Network parameters

In [4]:
for i in np.arange(0, 3):
    Setting_no = i

    Max_length = 1000
    learning_rate = 0.001
    batch_size = 150
    epochs = 100
    aa_vocab_size = 25
    dna_vocab_size = 67


    hidden_size_enc = int(Settings['Enc hidden size'][Setting_no])
    hidden_size_enc_aa = int(Settings['Enc hidden size'][Setting_no])
    embedding_size_enc = int(Settings['Enc Embedding size'][Setting_no])
    embedding_size_dec = int(Settings['Dec Embedding size'][Setting_no])
    Dense_layer_size = int(Settings['Dense Layer size'][Setting_no])
    Dense_layer_size_aa = int(Settings['Dense Layer size aa'][Setting_no])

    drop_rate = Settings['Drop rate'][Setting_no]
    drop_rate_aa = Settings['Drop rate aa'][Setting_no]

    input_sequence = Input(shape=(Max_length,))
    encod_emb = Embedding(input_dim= aa_vocab_size, output_dim = embedding_size_enc,trainable=True, mask_zero = True)
    embedding = encod_emb(input_sequence)

    encoder = Bidirectional(GRU(hidden_size_enc, return_sequences=True, return_state = True),
                            merge_mode="concat", weights=None)

    encoder_sequence, encoder_final_f, encoder_final_b  = encoder(embedding)

    encoder_final = Concatenate(axis=-1)([encoder_final_f, encoder_final_b])


    
    decoder_inputs = Input(shape=(Max_length -1, ))
    decoder_inputs_aa = Input(shape=(Max_length, ))

    dex=  Embedding(input_dim = dna_vocab_size, output_dim = embedding_size_dec, trainable=True, mask_zero = True)


    final_dex= dex(decoder_inputs)
    final_dex_aa =  encod_emb(decoder_inputs_aa)


    decoder = GRU(2*hidden_size_enc, return_sequences = True, return_state = True)
    decoder_aa =  GRU(2*hidden_size_enc_aa, return_sequences = True, return_state = True)

    decoder_sequence, decoder_final = decoder(final_dex, initial_state=encoder_final)
    decoder_sequence_aa, decoder_final_aa = decoder_aa(final_dex_aa, initial_state=encoder_final)


    attn_layer = Attention()
    attn_out = attn_layer([decoder_sequence, encoder_sequence])
    attn_out_aa = attn_layer([decoder_sequence_aa, encoder_sequence])

    decoder_concat_input = Concatenate(axis=-1)([decoder_sequence, attn_out]) #decoder_sequence, 
    decoder_concat_input_aa = Concatenate(axis=-1)([decoder_sequence_aa, attn_out_aa]) #decoder_sequence,


    Intermediate_layer = TimeDistributed(Dense(Dense_layer_size, activation='tanh'))
    Intermediate_layer_aa= TimeDistributed(Dense(Dense_layer_size_aa, activation='tanh'))

    Intemediate_output = Intermediate_layer(decoder_concat_input) #decoder_concat_input
    Intemediate_output_aa = Intermediate_layer_aa(decoder_concat_input_aa) #decoder_concat_input


    dropout_layer = Dropout(drop_rate)
    dropout_output = dropout_layer(Intemediate_output)

    dropout_layer_aa = Dropout(drop_rate_aa)
    dropout_output_aa = dropout_layer(Intemediate_output_aa)

    dense_layer = TimeDistributed(Dense(dna_vocab_size, activation='softmax'))
    logits = dense_layer(dropout_output)

    dense_layer_aa = TimeDistributed(Dense(aa_vocab_size, activation='softmax'))
    logits_aa = dense_layer_aa(dropout_output_aa)

    enc_dec_model = Model([input_sequence, decoder_inputs, decoder_inputs_aa], [logits, logits_aa])

    enc_dec_model.compile(loss=sparse_categorical_crossentropy,
                  optimizer=Adam(learning_rate = learning_rate),
                  metrics=['accuracy'])
    enc_dec_model.summary()
    
    checkpoint_path = "/Desktop/training_2Target/Round2/" + str(Setting_no) + "/cp.ckpt"
    checkpoint_dir = os.path.dirname(checkpoint_path)

    # Create a callback that saves the model's weights
    cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path,
                                                 save_weights_only=True,
                                                 verbose=1)

    early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", min_delta=0, patience = 3,
        verbose=0, mode="auto", baseline=None, restore_best_weights=False)
    ## Train the model
    model_results = enc_dec_model.fit([AA_tr[:,1:Max_length+1], Cds_tr[:,0:Max_length-1], AA_tr[:,0:Max_length]], 
                                      [Cds_tr[:,1:Max_length],  AA_tr[:,1:Max_length+1]], 
                                      batch_size= batch_size, 
                                      epochs= epochs, 
                                  validation_split=0.2, callbacks=[cp_callback, early_stop])
    
    name_history = 'Loss_Evolution/Round2/Combo'+ str(Setting_no) + '.csv'
    pd.DataFrame(model_results.history).to_csv(name_history)
    
    name_model = 'saved_models/Round2/EncDec_' + str(Setting_no)
    enc_dec_model.save(name_model)

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 1000)]       0           []                               
                                                                                                  
 embedding (Embedding)          (None, 1000, 235)    5875        ['input_1[0][0]',                
                                                                  'input_3[0][0]']                
                                                                                                  
 input_2 (InputLayer)           [(None, 999)]        0           []                               
                                                                                                  
 bidirectional (Bidirectional)  [(None, 1000, 120),  106920      ['embedding[0][0]']          

Epoch 5/100
133/133 [==============================] - ETA: 0s - loss: 0.4767 - time_distributed_2_loss: 0.4745 - time_distributed_3_loss: 0.0022 - time_distributed_2_accuracy: 0.4720 - time_distributed_3_accuracy: 0.9988
Epoch 5: saving model to /Desktop/training_2Target/Round2/0\cp.ckpt
133/133 [==============================] - 38s 288ms/step - loss: 0.4767 - time_distributed_2_loss: 0.4745 - time_distributed_3_loss: 0.0022 - time_distributed_2_accuracy: 0.4720 - time_distributed_3_accuracy: 0.9988 - val_loss: 0.4713 - val_time_distributed_2_loss: 0.4694 - val_time_distributed_3_loss: 0.0019 - val_time_distributed_2_accuracy: 0.4770 - val_time_distributed_3_accuracy: 0.9990
Epoch 6/100
133/133 [==============================] - ETA: 0s - loss: 0.4672 - time_distributed_2_loss: 0.4657 - time_distributed_3_loss: 0.0015 - time_distributed_2_accuracy: 0.4801 - time_distributed_3_accuracy: 0.9992
Epoch 6: saving model to /Desktop/training_2Target/Round2/0\cp.ckpt
133/133 [===============

Epoch 17/100
133/133 [==============================] - ETA: 0s - loss: 0.4497 - time_distributed_2_loss: 0.4494 - time_distributed_3_loss: 3.5982e-04 - time_distributed_2_accuracy: 0.4964 - time_distributed_3_accuracy: 0.9998
Epoch 17: saving model to /Desktop/training_2Target/Round2/0\cp.ckpt
133/133 [==============================] - 40s 304ms/step - loss: 0.4497 - time_distributed_2_loss: 0.4494 - time_distributed_3_loss: 3.5982e-04 - time_distributed_2_accuracy: 0.4964 - time_distributed_3_accuracy: 0.9998 - val_loss: 0.4511 - val_time_distributed_2_loss: 0.4505 - val_time_distributed_3_loss: 5.9944e-04 - val_time_distributed_2_accuracy: 0.4949 - val_time_distributed_3_accuracy: 0.9997
Epoch 18/100
133/133 [==============================] - ETA: 0s - loss: 0.4494 - time_distributed_2_loss: 0.4490 - time_distributed_3_loss: 3.5582e-04 - time_distributed_2_accuracy: 0.4969 - time_distributed_3_accuracy: 0.9998
Epoch 18: saving model to /Desktop/training_2Target/Round2/0\cp.ckpt
133/

Epoch 29/100
133/133 [==============================] - ETA: 0s - loss: 0.4463 - time_distributed_2_loss: 0.4461 - time_distributed_3_loss: 2.0136e-04 - time_distributed_2_accuracy: 0.5009 - time_distributed_3_accuracy: 0.9999
Epoch 29: saving model to /Desktop/training_2Target/Round2/0\cp.ckpt
133/133 [==============================] - 38s 286ms/step - loss: 0.4463 - time_distributed_2_loss: 0.4461 - time_distributed_3_loss: 2.0136e-04 - time_distributed_2_accuracy: 0.5009 - time_distributed_3_accuracy: 0.9999 - val_loss: 0.4486 - val_time_distributed_2_loss: 0.4482 - val_time_distributed_3_loss: 4.0794e-04 - val_time_distributed_2_accuracy: 0.4978 - val_time_distributed_3_accuracy: 0.9998
Epoch 30/100
133/133 [==============================] - ETA: 0s - loss: 0.4463 - time_distributed_2_loss: 0.4460 - time_distributed_3_loss: 2.3963e-04 - time_distributed_2_accuracy: 0.5011 - time_distributed_3_accuracy: 0.9999
Epoch 30: saving model to /Desktop/training_2Target/Round2/0\cp.ckpt
133/

Epoch 41/100
133/133 [==============================] - ETA: 0s - loss: 0.4439 - time_distributed_2_loss: 0.4438 - time_distributed_3_loss: 1.3852e-04 - time_distributed_2_accuracy: 0.5048 - time_distributed_3_accuracy: 0.9999
Epoch 41: saving model to /Desktop/training_2Target/Round2/0\cp.ckpt
133/133 [==============================] - 37s 276ms/step - loss: 0.4439 - time_distributed_2_loss: 0.4438 - time_distributed_3_loss: 1.3852e-04 - time_distributed_2_accuracy: 0.5048 - time_distributed_3_accuracy: 0.9999 - val_loss: 0.4471 - val_time_distributed_2_loss: 0.4468 - val_time_distributed_3_loss: 3.3459e-04 - val_time_distributed_2_accuracy: 0.4997 - val_time_distributed_3_accuracy: 0.9998
Epoch 42/100
133/133 [==============================] - ETA: 0s - loss: 0.4437 - time_distributed_2_loss: 0.4436 - time_distributed_3_loss: 1.5218e-04 - time_distributed_2_accuracy: 0.5051 - time_distributed_3_accuracy: 0.9999
Epoch 42: saving model to /Desktop/training_2Target/Round2/0\cp.ckpt
133/

...

Epoch 53/100
133/133 [==============================] - ETA: 0s - loss: 0.8941 - time_distributed_6_loss: 0.8903 - time_distributed_7_loss: 0.0039 - time_distributed_6_accuracy: 0.2488 - time_distributed_7_accuracy: 0.9967
Epoch 53: saving model to /Desktop/training_2Target/Round2/1\cp.ckpt
133/133 [==============================] - 36s 272ms/step - loss: 0.8941 - time_distributed_6_loss: 0.8903 - time_distributed_7_loss: 0.0039 - time_distributed_6_accuracy: 0.2488 - time_distributed_7_accuracy: 0.9967 - val_loss: 0.4664 - val_time_distributed_6_loss: 0.4656 - val_time_distributed_7_loss: 8.0604e-04 - val_time_distributed_6_accuracy: 0.4645 - val_time_distributed_7_accuracy: 0.9998


Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_7 (InputLayer)           [(None, 1000)]       0           []                               
                                                                                                  
 embedding_4 (Embedding)        (None, 1000, 83)     2075        ['input_7[0][0]',                
                                                                  'input_9[0][0]']                
                                                                                                  
 input_8 (InputLayer)           [(None, 999)]        0           []                               
                                                                                                  
 bidirectional_2 (Bidirectional  [(None, 1000, 386),  321924     ['embedding_4[0][0]']      

Epoch 5/100
133/133 [==============================] - ETA: 0s - loss: 0.4846 - time_distributed_10_loss: 0.4665 - time_distributed_11_loss: 0.0182 - time_distributed_10_accuracy: 0.4719 - time_distributed_11_accuracy: 0.9875
Epoch 5: saving model to /Desktop/training_2Target/Round2/2\cp.ckpt
133/133 [==============================] - 61s 457ms/step - loss: 0.4846 - time_distributed_10_loss: 0.4665 - time_distributed_11_loss: 0.0182 - time_distributed_10_accuracy: 0.4719 - time_distributed_11_accuracy: 0.9875 - val_loss: 0.4595 - val_time_distributed_10_loss: 0.4586 - val_time_distributed_11_loss: 8.1995e-04 - val_time_distributed_10_accuracy: 0.4829 - val_time_distributed_11_accuracy: 0.9997
Epoch 6/100
133/133 [==============================] - ETA: 0s - loss: 0.4773 - time_distributed_10_loss: 0.4642 - time_distributed_11_loss: 0.0131 - time_distributed_10_accuracy: 0.4747 - time_distributed_11_accuracy: 0.9902
Epoch 6: saving model to /Desktop/training_2Target/Round2/2\cp.ckpt
133/

Epoch 17/100
133/133 [==============================] - ETA: 0s - loss: 0.4617 - time_distributed_10_loss: 0.4561 - time_distributed_11_loss: 0.0056 - time_distributed_10_accuracy: 0.4859 - time_distributed_11_accuracy: 0.9944
Epoch 17: saving model to /Desktop/training_2Target/Round2/2\cp.ckpt
133/133 [==============================] - 60s 452ms/step - loss: 0.4617 - time_distributed_10_loss: 0.4561 - time_distributed_11_loss: 0.0056 - time_distributed_10_accuracy: 0.4859 - time_distributed_11_accuracy: 0.9944 - val_loss: 0.4516 - val_time_distributed_10_loss: 0.4509 - val_time_distributed_11_loss: 6.5953e-04 - val_time_distributed_10_accuracy: 0.4928 - val_time_distributed_11_accuracy: 0.9998
Epoch 18/100
133/133 [==============================] - ETA: 0s - loss: 0.4613 - time_distributed_10_loss: 0.4558 - time_distributed_11_loss: 0.0056 - time_distributed_10_accuracy: 0.4864 - time_distributed_11_accuracy: 0.9945
Epoch 18: saving model to /Desktop/training_2Target/Round2/2\cp.ckpt


FailedPreconditionError: {{function_node __wrapped__MergeV2Checkpoints_device_/job:localhost/replica:0/task:0/device:CPU:0}} Failed to rename: saved_models/Round2/EncDec_2\variables\variables_temp/part-00000-of-00001.data-00000-of-00001 to: saved_models/Round2/EncDec_2\variables\variables.data-00000-of-00001 : The process cannot access the file because it is being used by another process.
; Broken pipe [Op:MergeV2Checkpoints]

In [6]:
name_history = 'Loss_Evolution/Round2/Combo'+ str(Setting_no) + '.csv'
pd.DataFrame(model_results.history).to_csv(name_history)

name_model = 'saved_models/Round2/EncDec_' + str(Setting_no)
enc_dec_model.save(name_model)

#### Length and its plot

In [ ]:
length_aa = []
for i in range(len(AA_seq_tokenized)):
    length_aa.append(len(AA_seq_tokenized[i]))
print(max(length_aa))

sns.histplot(length_aa)


In [ ]:
 Cds_ts[0,0]

In [5]:
Setting_no

2